# 🚀 梯度提升樹（Gradient Boosted Decision Trees, GBDT）

梯度提升樹是另一種廣泛應用的決策樹集成方法。與隨機森林平行建立多棵獨立樹不同，梯度提升樹採用**循序（Sequential）建立決策樹**的方式，每一棵新樹都專注於**修正前一棵樹產生的預測誤差**。

### 💡 GBDT 核心特性：
1. **預剪枝深度較淺**：通常使用深度很淺的決策樹（如 `max_depth = 1 ~ 5`），這使得每棵弱學習器速度快且記憶體佔用低。
2. **結合學習率（Learning Rate）**：學習率控制每棵樹修正先前誤差的強度。學習率越低，需要越多棵樹來構建模型，但泛化能力往往更好。
3. **高精準度**：在 Kaggle 等機器學習競賽中，GBDT 及其變體（XGBoost, LightGBM, CatBoost）常常是表格數據（Tabular Data）表現最強大的模型。

In [ ]:
# 1. 預設參數的 GradientBoostingClassifier
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier

cancer = load_breast_cancer()
X_train, X_test, y_train, y_test = train_test_split(
    cancer.data, cancer.target, stratify=cancer.target, random_state=0
)

gbrt = GradientBoostingClassifier(random_state=0)
gbrt.fit(X_train, y_train)

print(f"預設 GBDT 訓練集準確率：{gbrt.score(X_train, y_train):.3f}")
print(f"預設 GBDT 測試集準確率：{gbrt.score(X_test, y_test):.3f}")

### 2. GBDT 預剪枝與學習率調參

預設的 GBDT 在訓練集上的準確率高達 100%（1.000），這可能代表模型有輕微過擬合的趨勢。

我們可以透過以下兩種方式進行強度調控以減少過擬合：
1. **限制最大深度（`max_depth=1`）**：使每棵決策樹僅進行一次切割。
2. **調低學習率（`learning_rate=0.01`）**：減緩每棵樹的更新幅度。

In [ ]:
# 方式 A: 限制最大深度 (max_depth=1)
gbrt_depth1 = GradientBoostingClassifier(random_state=0, max_depth=1)
gbrt_depth1.fit(X_train, y_train)
print(f"預剪枝 (max_depth=1) 訓練集準確率：{gbrt_depth1.score(X_train, y_train):.3f}")
print(f"預剪枝 (max_depth=1) 測試集準確率：{gbrt_depth1.score(X_test, y_test):.3f}")

# 方式 B: 調低學習率 (learning_rate=0.01)
gbrt_lr = GradientBoostingClassifier(random_state=0, learning_rate=0.01)
gbrt_lr.fit(X_train, y_train)
print(f"調低學習率 (learning_rate=0.01) 訓練集準確率：{gbrt_lr.score(X_train, y_train):.3f}")
print(f"調低學習率 (learning_rate=0.01) 測試集準確率：{gbrt_lr.score(X_test, y_test):.3f}")

### 3. GBDT 的特徵重要性（Feature Importances）視覺化

與隨機森林類似，GBDT 同樣提供特徵重要性評分。但因為 GBDT 的樹比較淺，通常只會關注少數關鍵特徵：

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

# 載入中文字型
my_font = fm.FontProperties(fname="ChineseFont.ttf")

zh_feature_names = [
    "平均半徑", "平均紋理", "平均周長", "平均面積", "平均平滑度",
    "平均緊密度", "平均凹陷度", "平均凹陷點數", "平均對稱性", "平均分形維度",
    "半徑標準誤", "紋理標準誤", "周長標準誤", "面積標準誤", "平滑度標準誤",
    "緊密度標準誤", "凹陷度標準誤", "凹陷點數標準誤", "對稱性標準誤", "分形維度標準誤",
    "最大半徑", "最大紋理", "最大周長", "最大面積", "最大平滑度",
    "最大緊密度", "最大凹陷度", "最大凹陷點數", "最大對稱性", "最大分形維度"
]

n_features = cancer.data.shape[1]
plt.figure(figsize=(10, 8))
plt.barh(range(n_features), gbrt_depth1.feature_importances_, align="center")
plt.yticks(np.arange(n_features), zh_feature_names, fontproperties=my_font)
plt.xlabel("特徵重要性 (Feature Importance)", fontproperties=my_font, fontsize=12)
plt.ylabel("特徵名稱", fontproperties=my_font, fontsize=12)
plt.title("乳癌資料集 - 梯度提升樹 (GBDT, max_depth=1) 特徵重要性分佈", fontproperties=my_font, fontsize=14)
plt.ylim(-1, n_features)
plt.tight_layout()
plt.show()